In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from sklearn.datasets import make_blobs
from sklearn.metrics import accuracy_score
from IPython.display import HTML

plt.style.use('dark_background')
plt.rcParams.update({
    "figure.facecolor":  (0.12, 0.12, 0.12, 1),
    "axes.facecolor": (0.12, 0.12, 0.12, 1),
})

In [ ]:
X, y = make_blobs(n_samples=100, n_features=2, centers=2, random_state=0)
y = y.reshape((y.shape[0], 1))

print('dimensions de X:', X.shape)
print('dimensions de y:', y.shape)

In [ ]:
def initialisation(X):
    W = np.random.randn(X.shape[1], 1)
    b = np.random.randn(1)
    return (W, b)

def model(X, W, b):
    Z = X.dot(W) + b
    A = 1 / (1 + np.exp(-Z))
    return A

def log_loss(A, y):
    return 1 / len(y) * np.sum(-y * np.log(A) - (1 - y) * np.log(1 - A))

def gradients(A, X, y):
    dW = 1 / len(y) * np.dot(X.T, A - y)
    db = 1 / len(y) * np.sum(A - y)
    return (dW, db)

def update(dW, db, W, b, learning_rate):
    W = W - learning_rate * dW
    b = b - learning_rate * db
    return (W, b)

def train_with_history(X, y, learning_rate=0.1, n_iter=400):
    W, b = initialisation(X)
    history = []

    for i in range(n_iter):
        A = model(X, W, b)
        loss = log_loss(A, y)
        dW, db = gradients(A, X, y)
        W, b = update(dW, db, W, b, learning_rate)
        history.append({'W': W.copy(), 'b': b.copy(), 'loss': loss, 'iter': i})

    return history

In [ ]:
history = train_with_history(X, y, learning_rate=0.1, n_iter=400)
print(f"Loss initiale: {history[0]['loss']:.4f}")
print(f"Loss finale:   {history[-1]['loss']:.4f}")

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(14, 4))
fig.patch.set_facecolor((0.12, 0.12, 0.12))
for ax in axes:
    ax.set_facecolor((0.12, 0.12, 0.12))

# 1 frame sur 10 → 40 frames au total
step = 10
frames_idx = list(range(0, len(history), step))

colors = np.where(y.flatten() == 1, '#f0e442', '#009e73')
z_range = np.linspace(-5, 5, 200)
sigmoid_curve = 1 / (1 + np.exp(-z_range))
losses = [h['loss'] for h in history]

def animate(frame_idx):
    W = history[frame_idx]['W']
    b = history[frame_idx]['b']
    it = history[frame_idx]['iter']

    for ax in axes:
        ax.cla()
        ax.set_facecolor((0.12, 0.12, 0.12))

    # Frontière de Décision
    ax0 = axes[0]
    ax0.scatter(X[:, 0], X[:, 1], c=colors, s=25, zorder=3)
    x1_range = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 100)
    if W[1] != 0:
        ax0.plot(x1_range, (-W[0] * x1_range - b) / W[1], color='#f5a623', linewidth=2)
    ax0.set_title('Frontiere de Décision', color='white', fontsize=10)
    ax0.set_xlabel('x1', color='white'); ax0.set_ylabel('x2', color='white')
    ax0.tick_params(colors='white')
    ax0.set_xlim(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5)
    ax0.set_ylim(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5)

    # Sigmoid
    ax1 = axes[1]
    ax1.plot(z_range, sigmoid_curve, color='#f5a623', linewidth=2)
    Z_pts = (X.dot(W) + b).flatten()
    A_pts = (1 / (1 + np.exp(-Z_pts)))
    y_flat = y.flatten()
    for i in range(len(X)):
        ax1.plot([Z_pts[i], Z_pts[i]], [y_flat[i], A_pts[i]], color='red', alpha=0.4, linewidth=0.6)
    ax1.scatter(Z_pts, A_pts, c=colors, s=20, zorder=4)
    ax1.set_title('Sigmoid', color='white', fontsize=10)
    ax1.set_xlabel('z', color='white'); ax1.set_ylabel('σ(z)', color='white')
    ax1.tick_params(colors='white')
    ax1.set_xlim(-5, 5); ax1.set_ylim(-0.05, 1.05)

    # Fonction Coût
    ax2 = axes[2]
    ax2.plot(range(frame_idx + 1), losses[:frame_idx + 1], color='red', linewidth=2)
    ax2.set_title('Fonction Coût', color='white', fontsize=10)
    ax2.set_xlabel('iteration', color='white'); ax2.set_ylabel('loss', color='white')
    ax2.tick_params(colors='white')
    ax2.set_xlim(0, len(history)); ax2.set_ylim(0, losses[0] * 1.05)

    fig.suptitle(f'Iteration {it}', color='white', fontsize=12)
    return axes

anim = animation.FuncAnimation(fig, animate, frames=frames_idx, interval=100, repeat=False)
plt.close()
HTML(anim.to_jshtml())

In [ ]:
# Optionnel : sauvegarder en MP4 (nécessite ffmpeg installé)
# Writer = animation.writers['ffmpeg']
# writer = Writer(fps=30, metadata=dict(artist='Me'), bitrate=3200)
# anim.save('animation.mp4', writer=writer)
# print('Sauvegardé : animation.mp4')

In [ ]:
Z = np.linspace(-10, 10, 300)
A = 1 / (1 + np.exp(-Z))

plt.figure(figsize=(8, 4))
plt.plot(Z, A, color='#f5a623', linewidth=2)
plt.axvline(0, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
plt.axhline(0.5, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
plt.scatter([0], [0.5], color='red', zorder=5, label='Z=0 → A=0.5')
plt.title('Fonction Sigmoid', color='white')
plt.xlabel('Z', color='white')
plt.ylabel('A = σ(Z)', color='white')
plt.legend()
plt.ylim(-0.05, 1.05)
plt.show()